# Graph functions

In [ ]:
def print_graph(graph):
  """
  Prints a graph given by the adjacency list (or just a dictionary).
  """
  for v in graph:
    print(v, ": ", sep = "", end="")
    for u in graph[v]:
      print(u, end=" ")
    print()

def add_vertex(graph, vertex):
    if vertex not in graph:
        graph[vertex] = []

def add_arc(graph, arc):
    u, v = arc
    add_vertex(graph, u)
    add_vertex(graph, v)
    if v not in graph[u]:
        graph[u].append(v)

def add_edge(graph, edge):
    u, v = edge
    if u != v:
        add_arc(graph, (u, v))
        add_arc(graph, (v, u))
    else:
        raise ValueError('Loops are not allowed')

# Given functions (reading from / writing to a file)

In [ ]:
def graph_from_edges(filename, directed=0):
  """
  Reads a graph from a file in which each line contains two (or more) words -
  - the description of an arc (or edge) or just one word (the vertex name).
  Variable filename contains full file path.
  """
  graph = {}
  with open(filename, "r") as f:  # open for reading
    for line in f:
      words = line.strip().split()
      if len(words) == 1:         # the vertex name
        add_vertex(graph, words[0])
      elif len(words) >= 2:       # the arc/edge descriptions - use just first two
        if directed:
          add_arc(graph, (words[0], words[1]))
        else:
          add_edge(graph, (words[0], words[1]))
  return graph

def graph_to_neighbourlist(graph, filename):
  """
  Writes a graph given by the adjacency list (or just a dictionary) to the textfile.
  """
  with open(filename, "w") as f:  # open for writing
    for v in graph:
      line = f"{v}:"
      for u in graph[v]:
        line += f" {u}"
      line += "\n"
      f.write(line)

# Homework: graph_to_edges and graph_from_neighbourlist

In [ ]:
def graph_to_edges(graph, filename, directed=0):
  """
  Writes a graph given by the adjacency list (or just a dictionary) to the
  textfile, one edge (arc) per line: "u v". Vertices which do not occur in
  any written edge (isolated ones) are written as a single word in a line.
  If directed is false, every edge is written only once.
  """
  written = set()      # undirected: pairs already written
  used = set()         # vertices which occurred in some line
  with open(filename, "w") as f:
    for u in graph:
      for v in graph[u]:
        if not directed:
          if (v, u) in written:       # the edge was already written as "v u"
            continue
          written.add((u, v))
        f.write(f"{u} {v}\n")
        used.add(u)
        used.add(v)
    for v in graph:                   # isolated vertices
      if v not in used:
        f.write(f"{v}\n")

In [ ]:
def graph_from_neighbourlist(filename):
  """
  Reads a graph from a file in which each line has the form
  "v: u1 u2 u3 ..." (the neighbour list of v; "v:" for no neighbours).
  Returns the graph as a dictionary of lists.
  """
  graph = {}
  with open(filename, "r") as f:
    for line in f:
      line = line.strip()
      if not line:
        continue
      if line.endswith(":"):          # no neighbours: the line is just "v:"
        line += " "                   # so that split(": ") still gives two parts
      parts = line.split(": ")        # first split: vertex | neighbours
      v = parts[0]
      add_vertex(graph, v)
      for u in parts[1].split():      # second split: neighbours
        add_arc(graph, (v, u))
  return graph

# Testing

In [ ]:
%%writefile weighted0.txt
A B 3
A E 10
B C 26
B D 12
C D 17
C F 13
C G 14
D E 7
D F 15
E F 8
E H 4
F G 9
F H 6
G H 16
G I 11

In [ ]:
graph = graph_from_edges("weighted0.txt")
graph_to_neighbourlist(graph, "neighbourlist.txt")
g2 = graph_from_neighbourlist("neighbourlist.txt")
assert g2 == graph
print_graph(g2)

In [ ]:
graph_to_edges(graph, "edges_out.txt")
g3 = graph_from_edges("edges_out.txt")
# the order of neighbours may differ, so compare as sets
assert {v: set(u) for v, u in g3.items()} == {v: set(u) for v, u in graph.items()}
print_graph(g3)

In [ ]:
# directed graph with an isolated vertex
d = {"a": ["b", "c"], "b": [], "c": ["a"], "f": []}
graph_to_edges(d, "d_edges.txt", directed=1)
graph_to_neighbourlist(d, "d_nl.txt")
assert graph_from_edges("d_edges.txt", directed=1) == {"a": ["b", "c"], "b": [], "c": ["a"], "f": []}
assert graph_from_neighbourlist("d_nl.txt") == d